Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun dashboard pemantauan pesanan real-time untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (prototype) pipeline streaming yang mensimulasikan kebutuhan ini.

In [1]:
# Persiapan.
import os
import shutil
import time
from datetime import datetime
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window

# Inisialisasi SparkSession.
spark = SparkSession.builder \
    .appName("Tugas10_Naufal_Streaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 08:05:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 08:05:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


A. Menyiapkan Streaming Source (bobot 15%).

Definisikan skema eksplisit (StructType) untuk data pesanan dengan kolom: pesanan_id (string), nama_cabang (string), nama_menu (string), qty (integer), harga_satuan (integer), waktu_pesan (timestamp). Buat folder simulasi streaming baru bernama tugas10_stream, lalu definisikan readStream dari folder tersebut.

In [2]:
# 1. Siapkan folder simulasi lokal.
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Jembatan ke file lokal.
path_tugas_stream = "file://" + os.path.abspath(folder_tugas)

# 2. Definisikan skema eksplisit.
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# 3. Definisikan readStream.
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(path_tugas_stream)

print("Streaming source Bagian A siap.")

Streaming source Bagian A siap.


B. Agregasi per Cabang (bobot 20%).

Tambahkan kolom turunan total_harga (qty x harga_satuan) pada df_stream, lalu buat agregasi groupBy("nama_cabang") yang menghitung total total_harga dan jumlah pesanan. Jalankan sebagai streaming query dengan sink memory, queryName="agregasi_cabang".

In [3]:
# Tambahkan kolom turunan total_harga.
df_stream_dengan_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per cabang
agregasi_cabang = df_stream_dengan_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_penjualan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Jalankan query memory sink.
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query Bagian B (agregasi_cabang) berjalan.")

Streaming query Bagian B (agregasi_cabang) berjalan.


C. Agregasi Berbasis Window Waktu (bobot 20%).

Buat streaming query kedua (definisikan ulang readStream dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan. Gunakan sink memory, queryName="agregasi_waktu".

In [4]:
# Sumber streaming kedua dari folder yang sama.
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(path_tugas_stream)

df_stream_waktu_total = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi rentang window waktu 2 menit.
agregasi_waktu = df_stream_waktu_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_penjualan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Jalankan query memory sink kedua.
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query Bagian C (agregasi_waktu) berjalan.")

Streaming query Bagian C (agregasi_waktu) berjalan.


D. Menguji dengan Data Simulasi (bobot 25%).

Kirim minimal 3 batch data (masing-masing batch berupa satu berkas CSV baru dengan jeda time.sleep() di antaranya) ke folder tugas10_stream, dengan kombinasi nama_cabang yang bervariasi. Setelah setiap batch, tampilkan hasil agregasi_cabang untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [5]:
# --- BATCH 1 ---
waktu_1 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P101", "P102"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Semarang"],
    "nama_menu": ["Espresso", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [18000, 20000],
    "waktu_pesan": [waktu_1, waktu_1]
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 terkirim. Menunggu pemrosesan...")
time.sleep(6)
print("=== HASIL AGREGASI CABANG SETELAH BATCH 1 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_penjualan DESC").show(truncate=False)

# --- BATCH 2 ---
waktu_2 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P103", "P104"],
    "nama_cabang": ["Kopi Nusantara Yogyakarta", "Kopi Nusantara Magelang"],
    "nama_menu": ["Cafe Latte", "Cappuccino"],
    "qty": [2, 3],
    "harga_satuan": [25000, 25000],
    "waktu_pesan": [waktu_2, waktu_2]
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 terkirim. Menunggu pemrosesan...")
time.sleep(6)
print("=== HASIL AGREGASI CABANG SETELAH BATCH 2 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_penjualan DESC").show(truncate=False)

# --- BATCH 3 ---
waktu_3 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P105", "P106"],
    "nama_cabang": ["Kopi Nusantara Semarang", "Kopi Nusantara Yogyakarta"],
    "nama_menu": ["Caramel Macchiato", "Kopi Tubruk"],
    "qty": [1, 2],
    "harga_satuan": [30000, 15000],
    "waktu_pesan": [waktu_3, waktu_3]
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 terkirim. Menunggu pemrosesan...")
time.sleep(6)
print("=== HASIL AGREGASI CABANG SETELAH BATCH 3 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_penjualan DESC").show(truncate=False)

# Hasil window waktu 2 menit
print("=== HASIL AGREGASI WINDOW WAKTU (2 MENIT) ===")
spark.sql("SELECT window.start, window.end, total_penjualan, jumlah_pesanan FROM agregasi_waktu").show(truncate=False)

Batch 1 terkirim. Menunggu pemrosesan...
=== HASIL AGREGASI CABANG SETELAH BATCH 1 ===
+-----------------------+---------------+--------------+
|nama_cabang            |total_penjualan|jumlah_pesanan|
+-----------------------+---------------+--------------+
|Kopi Nusantara Magelang|36000          |1             |
|Kopi Nusantara Semarang|20000          |1             |
+-----------------------+---------------+--------------+

Batch 2 terkirim. Menunggu pemrosesan...
=== HASIL AGREGASI CABANG SETELAH BATCH 2 ===
+-------------------------+---------------+--------------+
|nama_cabang              |total_penjualan|jumlah_pesanan|
+-------------------------+---------------+--------------+
|Kopi Nusantara Magelang  |111000         |2             |
|Kopi Nusantara Yogyakarta|50000          |1             |
|Kopi Nusantara Semarang  |20000          |1             |
+-------------------------+---------------+--------------+

Batch 3 terkirim. Menunggu pemrosesan...
=== HASIL AGREGASI CABANG SE

E. Sink Berbasis Berkas (Tantangan) (bobot 20%)

Selain memory sink, Structured Streaming juga mendukung menulis hasil DataFrame mentah (bukan hasil agregasi) ke berkas secara berkelanjutan. Buat streaming query ketiga yang membaca dari df_stream, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal tugas10_output dalam format Parquet, menggunakan:

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (!ls tugas10_output) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan spark.read.parquet().

In [6]:
# 1. Siapkan folder output dan checkpoint lokal
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

path_output = "file://" + os.path.abspath(folder_output)
path_checkpoint = "file://" + os.path.abspath(folder_checkpoint)

# 2. Definisikan streaming query ketiga yang menulis data mentah ke Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", path_output) \
    .option("checkpointLocation", path_checkpoint) \
    .outputMode("append") \
    .start()

print("Streaming query Bagian E (Parquet Sink) berjalan.")

# 3. Kirim Batch 4
waktu_4 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P107", "P108"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Semarang"],
    "nama_menu": ["Nasi Goreng", "Es Kopi"],
    "qty": [1, 2],
    "harga_satuan": [25000, 15000],
    "waktu_pesan": [waktu_4, waktu_4]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu berkas Parquet selesai ditulis...")
time.sleep(8)

# 4. Tampilkan berkas Parquet di disk lokal
print("Struktur berkas pada folder tugas10_output:")
get_ipython().system('ls -lh tugas10_output')

# 5. Baca kembali secara batch untuk verifikasi keutuhan data
df_verifikasi_parquet = spark.read.parquet(path_output)
print("\nVerifikasi pembacaan ulang berkas Parquet:")
df_verifikasi_parquet.show(truncate=False)

Streaming query Bagian E (Parquet Sink) berjalan.
Batch 4 dikirim. Menunggu berkas Parquet selesai ditulis...


Struktur berkas pada folder tugas10_output:
total 20K
-rw-r--r-- 1 caitlyn caitlyn 1,9K Nov 11 08:27 part-00000-cf652b5c-aa43-4c7e-88b2-cf34f9d01d4d-c000.snappy.parquet
-rw-r--r-- 1 caitlyn caitlyn 1,9K Nov 11 08:27 part-00001-627d2089-9f28-45fc-9d3c-5de7f2ec6b69-c000.snappy.parquet
-rw-r--r-- 1 caitlyn caitlyn 1,9K Nov 11 08:27 part-00002-3e447ee4-aeef-4cf8-b497-4c344db7c6db-c000.snappy.parquet
-rw-r--r-- 1 caitlyn caitlyn 1,9K Nov 11 08:27 part-00003-867e0735-8721-4656-a16e-f454bea44783-c000.snappy.parquet
drwxr-xr-x 2 caitlyn caitlyn 4,0K Nov 11 08:27 _spark_metadata

Verifikasi pembacaan ulang berkas Parquet:
+----------+-------------------------+-----------------+---+------------+-------------------+
|pesanan_id|nama_cabang              |nama_menu        |qty|harga_satuan|waktu_pesan        |
+----------+-------------------------+-----------------+---+------------+-------------------+
|P103      |Kopi Nusantara Yogyakarta|Cafe Latte       |2  |25000       |2026-11-11 08:07:24|
|P1

In [7]:
# Menghentikan seluruh query dan mematikan session.
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()

print("Seluruh streaming query berhasil dihentikan dan SparkSession ditutup.")

Seluruh streaming query berhasil dihentikan dan SparkSession ditutup.
